# TD Python — Introduction au Web Scraping  

## Objectifs
- comprendre le rôle de `requests` et `BeautifulSoup`
- extraire des données avec des sélecteurs CSS
- structurer les résultats avec `list[dict]`
- charger les résultats dans `pandas`
- exporter en CSV
- identifier les limites d’un code simple et procédural

## Rappel
Le scraping doit respecter les règles d’usage du site, éviter la surcharge serveur, et ne pas collecter de données sensibles sans autorisation.


## Plan
### Données fictives
1. Lire un HTML simple  
2. Extraire un produit  
3. Extraire plusieurs produits  
4. Simuler plusieurs pages  
5. Exporter


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time


# Scraping de données fictives

Dans cette partie, aucune requête Internet n’est nécessaire.  
On travaille sur des chaînes HTML écrites directement dans le notebook.


In [ ]:
product_html = '''
<html>
    <body>
        <div class="product">
            <h1 class="title">Clavier mécanique</h1>
            <span class="price">89.99</span>
            <span class="rating">4.7</span>
        </div>
    </body>
</html>
'''

catalog_html = '''
<html>
    <body>
        <div class="product-card">
            <h2 class="title">Clavier mécanique</h2>
            <span class="price">89.99</span>
            <span class="rating">4.7</span>
        </div>
        <div class="product-card">
            <h2 class="title">Souris ergonomique</h2>
            <span class="price">49.50</span>
            <span class="rating">4.4</span>
        </div>
        <div class="product-card">
            <h2 class="title">Écran 27 pouces</h2>
            <span class="price">229.90</span>
            <span class="rating">4.8</span>
        </div>
    </body>
</html>
'''

article_html = '''
<html>
    <body>
        <article>
            <h1>Les bases du web scraping</h1>
            <p class="author">Awa Diop</p>
            <div class="content">Contenu de l'article...</div>
        </article>
    </body>
</html>
'''


## A.1 Explorer une page HTML fictive

In [ ]:
soup = BeautifulSoup(product_html, "html.parser")
print(soup.prettify())


## A.2 Extraire un produit unique

In [ ]:
title = soup.select_one(".title").get_text(strip=True)
price = float(soup.select_one(".price").get_text(strip=True))
rating = float(soup.select_one(".rating").get_text(strip=True))

product = {
    "title": title,
    "price": price,
    "rating": rating
}
product


### Exercice 1
À partir de `article_html`, extrayez :
- `title`
- `author`
- `content`

et stockez le tout dans un dictionnaire `article_data`.


In [ ]:
# TODO
# soup_article = BeautifulSoup(article_html, "html.parser")
# article_data = {...}
# article_data


## A.3 Extraire plusieurs produits d’une même page

In [ ]:
soup_catalog = BeautifulSoup(catalog_html, "html.parser")
cards = soup_catalog.select(".product-card")

products = []
for card in cards:
    products.append({
        "title": card.select_one(".title").get_text(strip=True),
        "price": float(card.select_one(".price").get_text(strip=True)),
        "rating": float(card.select_one(".rating").get_text(strip=True))
    })

products


In [ ]:
df_products = pd.DataFrame(products)
df_products


### Exercice 2
Ajoutez une colonne `price_category` :
- `low` si prix < 50
- `medium` si 50 <= prix < 150
- `high` si prix >= 150


In [ ]:
# TODO
# df_products["price_category"] = ...
# df_products


## A.4 Simuler un mini site avec plusieurs pages

In [ ]:
FAKE_PAGES = {
    "http://fake-shop.local/page-1.html": '''
    <html><body>
        <div class="product-card">
            <h2 class="title">Clavier mécanique</h2>
            <span class="price">89.99</span>
            <span class="rating">4.7</span>
        </div>
        <div class="product-card">
            <h2 class="title">Souris ergonomique</h2>
            <span class="price">49.50</span>
            <span class="rating">4.4</span>
        </div>
        <a class="next" href="http://fake-shop.local/page-2.html">Page suivante</a>
    </body></html>
    ''',
    "http://fake-shop.local/page-2.html": '''
    <html><body>
        <div class="product-card">
            <h2 class="title">Écran 27 pouces</h2>
            <span class="price">229.90</span>
            <span class="rating">4.8</span>
        </div>
        <div class="product-card">
            <h2 class="title">Hub USB-C</h2>
            <span class="price">34.99</span>
            <span class="rating">4.1</span>
        </div>
    </body></html>
    '''
}


## A.5 Premier scraper volontairement simple

In [ ]:
def scrape_fake_shop(start_url):
    current_url = start_url
    results = []

    while current_url:
        html = FAKE_PAGES[current_url]
        soup = BeautifulSoup(html, "html.parser")

        cards = soup.select(".product-card")
        for card in cards:
            results.append({
                "title": card.select_one(".title").get_text(strip=True),
                "price": float(card.select_one(".price").get_text(strip=True)),
                "rating": float(card.select_one(".rating").get_text(strip=True)),
                "page_url": current_url
            })

        next_link = soup.select_one(".next")
        current_url = next_link["href"] if next_link else None

    return results

fake_data = scrape_fake_shop("http://fake-shop.local/page-1.html")
fake_data


In [ ]:
pd.DataFrame(fake_data)


### Exercice 3
Modifiez la fonction précédente pour calculer :
- `price_with_tax = price * 1.20`


In [ ]:
# TODO
# Ajouter price_with_tax au dictionnaire produit


## A.6 Export CSV

In [ ]:
df_fake = pd.DataFrame(fake_data)
df_fake.to_csv("fake_scraping_results.csv", index=False, encoding="utf-8")
print("Fichier exporté : fake_scraping_results.csv")
